# QUBO hamiltoniano

# COPILOT VQE

In [56]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit_aer.primitives import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from scipy.optimize import minimize

# =========================
# PUNTO 1: Hamiltoniano QUBO/Ising
# =========================
n_qubits = 4
coeffs = [1.0, 0.5, 0.5, -1.0]
operators = ["IIII", "IIIZ", "IIZI", "IIZZ"]
hamiltonian = SparsePauliOp(operators, coeffs=coeffs)

# =========================
# PUNTO 2: Ansatz
# =========================
ansatz = EfficientSU2(n_qubits, entanglement="linear", reps=2)
initial_params = np.random.random(ansatz.num_parameters)

# Backend + transpilation
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(ansatz)
isa_observable = hamiltonian.apply_layout(isa_circuit.layout)

# =========================
# PUNTO 3: VQE con ottimizzazione classica
# =========================
estimator = Estimator()

def cost_function(theta: np.ndarray) -> float:
    """Ritorna <H> per i parametri theta."""
    job = estimator.run([(isa_circuit, isa_observable, theta)])
    result = job.result()[0]
    return float(result.data.evs)

print("=== VQE ===")
initial_energy = cost_function(initial_params)
print(f"Energia iniziale: {initial_energy:.6f}")

opt_result = minimize(
    cost_function,
    x0=initial_params,
    method="COBYLA",
    options={"maxiter": 80}
)

best_theta = opt_result.x
best_energy = opt_result.fun
print(f"Energia finale ottimizzata: {best_energy:.6f}")
print(f"Convergenza: {opt_result.success}, msg: {opt_result.message}")

# =========================
# PUNTO 4: Grover (4 qubit) target = '1011'
# q3 q2 q1 q0 = 1 0 1 1
# =========================
grover = QuantumCircuit(4, 4)

# Sovrapposizione iniziale su tutti i qubit
grover.h([0, 1, 2, 3])

def oracle_target_1011(qc: QuantumCircuit):
    # target: q3=1, q2=0, q1=1, q0=1
    # Per marcare uno stato con un solo 0 (q2), faccio X su q2
    qc.x(2)

    # Multi-controlled Z usando H-MCX-H sul target q3 con controlli q0,q1,q2
    qc.h(3)
    qc.mcx([0, 1, 2], 3)
    qc.h(3)

    # Undo
    qc.x(2)

def diffuser(qc: QuantumCircuit):
    qc.h([0, 1, 2, 3])
    qc.x([0, 1, 2, 3])

    qc.h(3)
    qc.mcx([0, 1, 2], 3)
    qc.h(3)

    qc.x([0, 1, 2, 3])
    qc.h([0, 1, 2, 3])

# Numero iterazioni Grover ~ floor(pi/4*sqrt(N/M)), N=16, M=1 => ~3
num_iterations = 3
for _ in range(num_iterations):
    oracle_target_1011(grover)
    diffuser(grover)

# Misura
grover.measure([0, 1, 2, 3], [0, 1, 2, 3])

print("\n=== GROVER ===")
print(grover.draw(fold=120))

# Esecuzione con Sampler
sampler = Sampler()
job = sampler.run([grover], shots=2048)

# In SamplerV2 i counts sono in data.meas.get_counts()
# counts = res.meas.get_counts()
res = job.result()[0]
bits = res.data.c              # BitArray
counts = bits.get_counts()     # dizionario tipo {'1011': 1500, ...}
print(counts)
print("Most probable:", max(counts, key=counts.get))

# Stato più frequente
# most_probable = max(counts, key=counts.get)
# print("Stato più probabile:", most_probable)

/tmp/ipykernel_55/711191727.py:22: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n_qubits, entanglement="linear", reps=2)


=== VQE ===
Energia iniziale: 1.225352
Energia finale ottimizzata: -0.883796
Convergenza: False, msg: Return from COBYLA because the objective function has been evaluated MAXFUN times.

=== GROVER ===
     ┌───┐          ┌───┐┌───┐               ┌───┐┌───┐               ┌───┐┌───┐               ┌───┐┌───┐          »
q_0: ┤ H ├───────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├────────────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├──────────»
     ├───┤       │  ├───┤├───┤            │  ├───┤├───┤            │  ├───┤├───┤            │  ├───┤├───┤          »
q_1: ┤ H ├───────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├────────────■──┤ H ├┤ X ├────────────■──┤ X ├┤ H ├──────────»
     ├───┤┌───┐  │  ├───┤├───┤┌───┐       │  ├───┤├───┤┌───┐       │  ├───┤├───┤┌───┐       │  ├───┤├───┤┌───┐     »
q_2: ┤ H ├┤ X ├──■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├─────»
     ├───┤├───┤┌─┴─┐├───┤├───┤├───┤┌───┐┌─┴─┐├───┤├───┤├───┤┌───┐┌─┴─┐├───┤├───┤├───┤┌───┐┌─┴─┐├─

## scenario reale

In [4]:
import numpy as np
from itertools import product
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

# =========================================================
# SCENARIO REALE: Channel Selection (4 candidati)
# x_i in {0,1} indica se seleziono il canale i
#
# Costo:
#   C(x) = sum_i q_i x_i + sum_{i<j} J_ij x_i x_j + A*(sum_i x_i - K)^2
#
# q_i  : "costo locale" (rumore, occupazione, error rate)
# J_ij : interferenza tra coppie di canali scelti insieme
# K    : numero desiderato di canali attivi
# A    : peso del vincolo
# =========================================================

n = 4
K = 2
A = 3.0  # peso vincolo cardinalità

# Esempio dati "realistici" (da telemetria)
# più basso = meglio
q = np.array([0.8, 0.2, 1.1, 0.5])

# Interferenze simmetriche (0 sulla diagonale)
J = np.array([
    [0.0, 1.2, 0.4, 0.9],
    [1.2, 0.0, 0.7, 0.3],
    [0.4, 0.7, 0.0, 1.0],
    [0.9, 0.3, 1.0, 0.0],
])

# ---------- Utility: costruzione QUBO ----------
# C(x)=const + sum_i a_i x_i + sum_{i<j} b_ij x_i x_j
# con vincolo A*(sum x - K)^2
# => a_i += A*(1 - 2K), b_ij += 2A, const += A*K^2
a = q.astype(float).copy()
b = np.zeros((n, n), dtype=float)
const = A * (K ** 2)

for i in range(n):
    a[i] += A * (1 - 2 * K)

for i in range(n):
    for j in range(i + 1, n):
        b[i, j] += J[i, j] + 2 * A

# ---------- Mapping QUBO -> Ising ----------
# x_i = (1 - Z_i)/2
# x_i x_j = 1/4 (1 - Z_i - Z_j + Z_i Z_j)

h = np.zeros(n, dtype=float)             # coefficienti Z_i
Jz = np.zeros((n, n), dtype=float)       # coefficienti Z_i Z_j
c0 = const

# termini lineari
for i in range(n):
    c0 += a[i] / 2
    h[i] += -a[i] / 2

# termini quadratici
for i in range(n):
    for j in range(i + 1, n):
        cij = b[i, j]
        c0 += cij / 4
        h[i] += -cij / 4
        h[j] += -cij / 4
        Jz[i, j] += cij / 4

# ---------- Costruzione SparsePauliOp ----------
paulis = []
coeffs = []

# termine costante
paulis.append("I" * n)
coeffs.append(c0)

# termini Z_i
for i in range(n):
    label = ["I"] * n
    # convenzione stringa Pauli: sinistra=qubit alto, destra=qubit basso
    label[n - 1 - i] = "Z"
    paulis.append("".join(label))
    coeffs.append(h[i])

# termini Z_i Z_j
for i in range(n):
    for j in range(i + 1, n):
        if abs(Jz[i, j]) > 1e-12:
            label = ["I"] * n
            label[n - 1 - i] = "Z"
            label[n - 1 - j] = "Z"
            paulis.append("".join(label))
            coeffs.append(Jz[i, j])

hamiltonian = SparsePauliOp(paulis, coeffs=np.array(coeffs, dtype=float))

# ---------- VQE (solo ottimizzazione parametri ansatz) ----------
ansatz = EfficientSU2(n, entanglement="linear", reps=2)
theta = np.random.random(ansatz.num_parameters)

backend = AerSimulator()
estimator = Estimator()

# transpile circuito
t_ansatz = transpile(ansatz, backend=backend, optimization_level=1)
obs = hamiltonian.apply_layout(t_ansatz.layout)

def energy(params):
    job = estimator.run([(t_ansatz, obs, params)])
    return float(job.result()[0].data.evs)

# ottimizzazione semplice random-restart manuale (robusta senza scipy)
best_e = float("inf")
best_theta = None
for _ in range(30):
    cand = np.random.uniform(0, 2*np.pi, size=ansatz.num_parameters)
    e = energy(cand)
    if e < best_e:
        best_e = e
        best_theta = cand

print("Energia minima stimata (VQE-like):", best_e)

# ---------- Estrazione soluzione operativa ----------
# Per n=4 in produzione useresti un post-process più avanzato;
# qui enumeriamo tutte le bitstring per scegliere il costo minimo reale.
def classical_cost(x_bits):
    x = np.array(x_bits, dtype=float)
    lin = np.dot(q, x)
    quad = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            quad += J[i, j] * x[i] * x[j]
    penalty = A * (np.sum(x) - K) ** 2
    return lin + quad + penalty

all_x = list(product([0, 1], repeat=n))
best_x = min(all_x, key=classical_cost)
best_cost = classical_cost(best_x)

print("Miglior configurazione canali (x0..x3):", best_x)
print("Costo reale minimo:", best_cost)

selected = [i for i, bit in enumerate(best_x) if bit == 1]
print("Canali da attivare:", selected)



/tmp/ipykernel_56/3846577559.py:104: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n, entanglement="linear", reps=2)
/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


Energia minima stimata (VQE-like): 3.1153781766961695
Miglior configurazione canali (x0..x3): (0, 1, 0, 1)
Costo reale minimo: 1.0
Canali da attivare: [1, 3]


## cambiamento Samu

In [2]:
import numpy as np
from itertools import product
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit import transpile

# ============================================================
# INPUT CYBER (esempio; in produzione arriva da JSON/SIEM/CMDB)
# ============================================================
# n canali/percorsi candidati
n = 6
K = 3  # quanti canali attivare

# feature normalizzate [0,1] per ogni canale i
cvss         = np.array([0.9, 0.3, 0.6, 0.2, 0.8, 0.4])
exploit_prob = np.array([0.8, 0.2, 0.7, 0.3, 0.9, 0.4])
patch_lag    = np.array([0.7, 0.4, 0.5, 0.2, 0.8, 0.3])
exposure     = np.array([0.9, 0.3, 0.6, 0.4, 0.7, 0.5])

latency      = np.array([0.6, 0.2, 0.4, 0.3, 0.7, 0.5])
jitter       = np.array([0.5, 0.3, 0.5, 0.2, 0.8, 0.4])
loss         = np.array([0.4, 0.2, 0.6, 0.3, 0.7, 0.5])

# compliance penalty (0 compliant, 1 non-compliant)
compliance   = np.array([0.0, 0.0, 0.5, 0.0, 1.0, 0.2])

# gruppi per anti-SPoF (idealmente max 1 selezione per gruppo)
# esempio: stesso provider/dominio di failure
groups = [
    [0, 2],      # gruppo A
    [1, 4, 5],   # gruppo B
    [3]          # gruppo C (singolo => irrilevante ma ok)
]

# matrice rischio correlato c_ij [0,1], simmetrica, diagonale zero
corr = np.array([
    [0.0, 0.2, 0.7, 0.1, 0.8, 0.4],
    [0.2, 0.0, 0.3, 0.2, 0.6, 0.5],
    [0.7, 0.3, 0.0, 0.2, 0.7, 0.4],
    [0.1, 0.2, 0.2, 0.0, 0.3, 0.2],
    [0.8, 0.6, 0.7, 0.3, 0.0, 0.6],
    [0.4, 0.5, 0.4, 0.2, 0.6, 0.0],
])

# pesi globali (tuning)
alpha = 1.0   # rischio intrinseco
beta  = 0.8   # rischio correlato
gamma = 0.4   # SLA cost
delta = 0.7   # compliance cost
A = 6.0       # vincolo cardinalità (sum x = K)
B = 3.0       # anti-SPoF gruppi

# pesi interni dei sotto-score
w_cvss, w_expl, w_patch, w_expo = 0.35, 0.25, 0.20, 0.20
w_lat, w_jit, w_loss = 0.5, 0.3, 0.2


# ============================================================
# 1) COSTRUZIONE SCORE LINEARE
# ============================================================
risk_intrinsic = (
    w_cvss * cvss +
    w_expl * exploit_prob +
    w_patch * patch_lag +
    w_expo * exposure
)

sla_cost = w_lat * latency + w_jit * jitter + w_loss * loss
comp_cost = compliance.copy()

linear_base = alpha * risk_intrinsic + gamma * sla_cost + delta * comp_cost
# linear_base[i] moltiplica x_i


# ============================================================
# 2) COSTRUZIONE QUBO x^T Q x + c
#    convenzione: Q simmetrica
# ============================================================
Q = np.zeros((n, n), dtype=float)
c = 0.0

# 2a) termini lineari base -> diagonale
for i in range(n):
    Q[i, i] += linear_base[i]

# 2b) rischio correlato beta * sum_{i<j} corr_ij x_i x_j
# con Q simmetrica: x^TQx contribuisce 2*Q_ij xixj => Q_ij += coeff/2
for i in range(n):
    for j in range(i + 1, n):
        coeff = beta * corr[i, j]
        Q[i, j] += coeff / 2.0
        Q[j, i] += coeff / 2.0

# 2c) vincolo cardinalità A*(sum x - K)^2
# espansione:
# A[sum_i x_i + 2 sum_{i<j} x_i x_j -2K sum_i x_i + K^2]
# = A[(1-2K)sum_i x_i + 2sum_{i<j}x_ix_j + K^2]
for i in range(n):
    Q[i, i] += A * (1 - 2 * K)
c += A * (K ** 2)

for i in range(n):
    for j in range(i + 1, n):
        coeff = 2 * A
        Q[i, j] += coeff / 2.0
        Q[j, i] += coeff / 2.0

# 2d) anti-SPoF per gruppo g: B*(sum_{i in g} x_i - 1)^2
for g in groups:
    if len(g) <= 1:
        continue
    for i in g:
        Q[i, i] += B * (1 - 2 * 1)  # B*(1-2)
    c += B * (1 ** 2)
    for idx_i in range(len(g)):
        for idx_j in range(idx_i + 1, len(g)):
            i = g[idx_i]
            j = g[idx_j]
            coeff = 2 * B
            Q[i, j] += coeff / 2.0
            Q[j, i] += coeff / 2.0


# ============================================================
# 3) QUBO -> SparsePauliOp (Ising)
# ============================================================
def qubo_to_sparse_pauli(Q: np.ndarray, c: float = 0.0, tol: float = 1e-12) -> SparsePauliOp:
    Q = np.array(Q, dtype=float)
    n = Q.shape[0]
    Qs = 0.5 * (Q + Q.T)

    const = float(c)
    h = np.zeros(n, dtype=float)
    Jzz = np.zeros((n, n), dtype=float)

    # diagonale
    for i in range(n):
        qii = Qs[i, i]
        const += 0.5 * qii
        h[i] += -0.5 * qii

    # off-diagonale
    for i in range(n):
        for j in range(i + 1, n):
            bij = 2.0 * Qs[i, j]  # coeff x_i x_j equivalente
            const += 0.25 * bij
            h[i] += -0.25 * bij
            h[j] += -0.25 * bij
            Jzz[i, j] += 0.25 * bij

    paulis, coeffs = [], []

    if abs(const) > tol:
        paulis.append("I" * n)
        coeffs.append(const)

    for i in range(n):
        if abs(h[i]) > tol:
            label = ["I"] * n
            label[n - 1 - i] = "Z"
            paulis.append("".join(label))
            coeffs.append(h[i])

    for i in range(n):
        for j in range(i + 1, n):
            if abs(Jzz[i, j]) > tol:
                label = ["I"] * n
                label[n - 1 - i] = "Z"
                label[n - 1 - j] = "Z"
                paulis.append("".join(label))
                coeffs.append(Jzz[i, j])

    if not paulis:
        paulis, coeffs = ["I" * n], [0.0]

    return SparsePauliOp(paulis, coeffs=np.array(coeffs, dtype=float))


hamiltonian = qubo_to_sparse_pauli(Q, c)


# ============================================================
# 4) VQE-like: minimizzazione energia (random search semplice)
# ============================================================
ansatz = EfficientSU2(n, entanglement="linear", reps=2)
backend = AerSimulator()
estimator = Estimator()

t_ansatz = transpile(ansatz, backend=backend, optimization_level=1)
obs = hamiltonian.apply_layout(t_ansatz.layout)

def energy(theta):
    job = estimator.run([(t_ansatz, obs, theta)])
    return float(job.result()[0].data.evs)

best_e = float("inf")
best_theta = None
for _ in range(60):
    th = np.random.uniform(0, 2*np.pi, ansatz.num_parameters)
    e = energy(th)
    if e < best_e:
        best_e = e
        best_theta = th

print("Best estimated quantum energy:", round(best_e, 6))


# ============================================================
# 5) Valutazione classica configurazioni + Top-3
# ============================================================
def qubo_cost_bits(bits, Q, c):
    x = np.array(bits, dtype=float)
    return float(x @ Q @ x + c)

def decode_selected(bits):
    return [i for i, b in enumerate(bits) if b == 1]

all_bits = list(product([0, 1], repeat=n))
scored = []
for bits in all_bits:
    cost = qubo_cost_bits(bits, Q, c)
    scored.append((cost, bits))

scored.sort(key=lambda t: t[0])

print("\nTop-3 configurazioni low-risk:")
for rank, (cost, bits) in enumerate(scored[:3], start=1):
    print(f"{rank}) bits={bits}  selected={decode_selected(bits)}  cost={cost:.6f}")

# opzionale: migliore soluzione che rispetta esattamente K
feasible = [(cost, bits) for (cost, bits) in scored if sum(bits) == K]
if feasible:
    best_feas = feasible[0]
    print("\nBest feasible (sum x = K):")
    print(f"bits={best_feas[1]} selected={decode_selected(best_feas[1])} cost={best_feas[0]:.6f}")

/tmp/ipykernel_55/1615734771.py:187: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n, entanglement="linear", reps=2)
/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


Best estimated quantum energy: 9.392691

Top-3 configurazioni low-risk:
1) bits=(1, 1, 0, 1, 0, 0)  selected=[0, 1, 3]  cost=2.207000
2) bits=(0, 1, 1, 1, 0, 0)  selected=[1, 2, 3]  cost=2.463000
3) bits=(1, 0, 0, 1, 0, 1)  selected=[0, 3, 5]  cost=2.708000

Best feasible (sum x = K):
bits=(1, 1, 0, 1, 0, 0) selected=[0, 1, 3] cost=2.207000


## creazione JSON input


In [13]:
import json
import random
from collections import defaultdict

random.seed(42)

N = 20
GROUP_COUNT = 40
TARGET_EDGES = 20000  # circa 18k-25k
MIN_DEGREE = 8

# -----------------------------
# Utility
# -----------------------------
def clamp01(x):
    return max(0.0, min(1.0, x))

def r4(x):
    return round(clamp01(x), 4)

def trunc_norm(mean, std, lo=0.0, hi=1.0):
    while True:
        v = random.gauss(mean, std)
        if lo <= v <= hi:
            return v

# -----------------------------
# Group distribution non-uniform
# -----------------------------
groups = [f"G{i:02d}" for i in range(GROUP_COUNT)]
# pesi non uniformi: alcuni gruppi molto grandi
weights = []
for i in range(GROUP_COUNT):
    if i < 8:
        w = random.uniform(2.5, 4.0)   # gruppi grandi
    elif i < 20:
        w = random.uniform(1.2, 2.2)   # medi
    else:
        w = random.uniform(0.5, 1.1)   # piccoli
    weights.append(w)

# normalizza e alloca cardinalità esatta N
w_sum = sum(weights)
raw_sizes = [N * w / w_sum for w in weights]
sizes = [int(x) for x in raw_sizes]
while sum(sizes) < N:
    idx = max(range(GROUP_COUNT), key=lambda i: raw_sizes[i] - sizes[i])
    sizes[idx] += 1

group_of = []
for gi, sz in enumerate(sizes):
    group_of.extend([groups[gi]] * sz)
random.shuffle(group_of)

# -----------------------------
# Generate channels
# -----------------------------
channels = []
for i in range(N):
    g = group_of[i]

    # "legacy factor": alcuni gruppi più legacy
    g_idx = int(g[1:])
    legacy_bias = 0.15 if g_idx < 10 else (0.05 if g_idx < 25 else -0.05)
    edge_bias = 0.12 if g_idx % 5 == 0 else 0.0

    cvss = trunc_norm(0.58 + legacy_bias * 0.2, 0.16)
    # outlier alti occasionali
    if random.random() < 0.04:
        cvss = max(cvss, random.uniform(0.9, 1.0))

    exploit_prob = clamp01(0.15 + 0.75 * cvss + random.gauss(0, 0.09))
    patch_lag = clamp01(0.20 + 0.60 * cvss + legacy_bias + random.gauss(0, 0.12))
    exposure = clamp01(0.25 + 0.55 * cvss + edge_bias + random.gauss(0, 0.14))

    # performance correlate
    latency = clamp01(0.15 + 0.45 * exposure + 0.20 * cvss + random.gauss(0, 0.10))
    jitter = clamp01(0.10 + 0.55 * latency + random.gauss(0, 0.08))
    loss = clamp01(0.08 + 0.45 * latency + 0.25 * jitter + random.gauss(0, 0.08))

    # compliance: molti 0, alcuni 0.2/0.5, pochi 1.0
    p = random.random()
    if p < 0.72:
        compliance = 0.0
    elif p < 0.88:
        compliance = 0.2
    elif p < 0.97:
        compliance = 0.5
    else:
        compliance = 1.0

    channels.append({
        "id": i,
        "cvss": r4(cvss),
        "exploit_prob": r4(exploit_prob),
        "patch_lag": r4(patch_lag),
        "exposure": r4(exposure),
        "latency": r4(latency),
        "jitter": r4(jitter),
        "loss": r4(loss),
        "compliance": r4(compliance),
        "group": g
    })

# -----------------------------
# Build correlations graph
# -----------------------------
group_to_ids = defaultdict(list)
for c in channels:
    group_to_ids[c["group"]].append(c["id"])

edges = set()  # store tuple(i,j) with i<j
degree = [0] * N

def add_edge(i, j, score):
    if i == j:
        return False
    a, b = (i, j) if i < j else (j, i)
    if (a, b) in edges:
        return False
    edges.add((a, b))
    degree[a] += 1
    degree[b] += 1
    return True

# Step 1: guarantee min degree >= MIN_DEGREE
all_ids = list(range(N))
for i in all_ids:
    attempts = 0
    while degree[i] < MIN_DEGREE and attempts < 4000:
        attempts += 1
        same_group = random.random() < 0.55
        gi = channels[i]["group"]
        if same_group and len(group_to_ids[gi]) > 1:
            j = random.choice(group_to_ids[gi])
            if j == i:
                continue
            base = 0.65
            score = r4(random.gauss(base, 0.12))
        else:
            # cross-group
            g2 = random.choice(groups)
            while g2 == gi or len(group_to_ids[g2]) == 0:
                g2 = random.choice(groups)
            j = random.choice(group_to_ids[g2])
            base = 0.25
            score = r4(random.gauss(base, 0.10))
        add_edge(i, j, score)

# Step 2: add random edges up to TARGET_EDGES
while len(edges) < TARGET_EDGES:
    same_group = random.random() < 0.45
    if same_group:
        g = random.choice(groups)
        ids = group_to_ids[g]
        if len(ids) < 2:
            continue
        i, j = random.sample(ids, 2)
        score = r4(random.gauss(0.65, 0.12))
    else:
        g1, g2 = random.sample(groups, 2)
        if not group_to_ids[g1] or not group_to_ids[g2]:
            continue
        i = random.choice(group_to_ids[g1])
        j = random.choice(group_to_ids[g2])
        score = r4(random.gauss(0.25, 0.10))
    add_edge(i, j, score)

# convert to list of objects
correlations = []
for (i, j) in sorted(edges):
    # score ricalcolato coerente con same/diff group
    if channels[i]["group"] == channels[j]["group"]:
        score = r4(random.gauss(0.65, 0.12))
    else:
        score = r4(random.gauss(0.25, 0.10))
    correlations.append({"i": i, "j": j, "score": score})

# -----------------------------
# Final JSON
# -----------------------------
data = {
    "channels": channels,
    "correlations": correlations,
    "constraints": {
        "K": 180,
        "weights": {
            "alpha": 1.0,
            "beta": 0.8,
            "gamma": 0.4,
            "delta": 0.7,
            "A": 6.0,
            "B": 3.0
        },
        "internal_weights": {
            "w_cvss": 0.35,
            "w_expl": 0.25,
            "w_patch": 0.20,
            "w_expo": 0.20,
            "w_lat": 0.5,
            "w_jit": 0.3,
            "w_loss": 0.2
        }
    }
}

# -----------------------------
# Validation
# -----------------------------
assert len(data["channels"]) == N
assert [c["id"] for c in data["channels"]] == list(range(N))
for c in data["channels"]:
    for k in ["cvss","exploit_prob","patch_lag","exposure","latency","jitter","loss","compliance"]:
        assert 0.0 <= c[k] <= 1.0
    assert c["group"] in groups

seen = set()
deg_check = [0]*N
for e in data["correlations"]:
    i, j, s = e["i"], e["j"], e["score"]
    assert 0 <= i < N and 0 <= j < N and i < j
    assert (i, j) not in seen
    seen.add((i, j))
    assert 0.0 <= s <= 1.0
    deg_check[i] += 1
    deg_check[j] += 1

assert all(d >= MIN_DEGREE for d in deg_check), "Alcuni nodi hanno degree < MIN_DEGREE"
assert 18000 <= len(data["correlations"]) <= 25000

with open("cyber_input_large.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)

print(f"Creato cyber_input_large.json con {len(channels)} channels e {len(correlations)} correlations.")

KeyboardInterrupt: 

## JSON fornito

In [9]:
import json
import numpy as np
from itertools import product
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit import transpile


# ============================================================
# 1) LOAD JSON
# ============================================================
def load_problem(path: str):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    channels = data["channels"]
    constraints = data["constraints"]
    correlations = data.get("correlations", [])

    n = len(channels)
    if n == 0:
        raise ValueError("channels vuoto")

    # ordina per id per sicurezza
    channels = sorted(channels, key=lambda c: c["id"])

    # verifica id consecutivi
    ids = [c["id"] for c in channels]
    if ids != list(range(n)):
        raise ValueError(f"Gli id dei canali devono essere consecutivi 0..{n-1}, trovato: {ids}")

    # estrazione feature
    def arr(key, default=0.0):
        return np.array([float(c.get(key, default)) for c in channels], dtype=float)

    cvss         = arr("cvss")
    exploit_prob = arr("exploit_prob")
    patch_lag    = arr("patch_lag")
    exposure     = arr("exposure")
    latency      = arr("latency")
    jitter       = arr("jitter")
    loss         = arr("loss")
    compliance   = arr("compliance")

    group_labels = [c.get("group", f"g_{i}") for i, c in enumerate(channels)]

    # groups da label
    group_map = {}
    for i, g in enumerate(group_labels):
        group_map.setdefault(g, []).append(i)
    groups = list(group_map.values())

    # vincoli/pesi
    K = int(constraints["K"])
    w = constraints["weights"]
    alpha = float(w["alpha"])
    beta  = float(w["beta"])
    gamma = float(w["gamma"])
    delta = float(w["delta"])
    A     = float(w["A"])
    B     = float(w["B"])

    # pesi interni (opzionali nel JSON, con default)
    internal = constraints.get("internal_weights", {})
    w_cvss = float(internal.get("w_cvss", 0.35))
    w_expl = float(internal.get("w_expl", 0.25))
    w_patch = float(internal.get("w_patch", 0.20))
    w_expo = float(internal.get("w_expo", 0.20))
    w_lat = float(internal.get("w_lat", 0.5))
    w_jit = float(internal.get("w_jit", 0.3))
    w_loss = float(internal.get("w_loss", 0.2))

    # matrice correlazioni
    corr = np.zeros((n, n), dtype=float)
    for item in correlations:
        i = int(item["i"]); j = int(item["j"]); s = float(item["score"])
        if i == j:
            continue
        corr[i, j] = s
        corr[j, i] = s

    return {
        "n": n, "K": K, "groups": groups,
        "cvss": cvss, "exploit_prob": exploit_prob, "patch_lag": patch_lag, "exposure": exposure,
        "latency": latency, "jitter": jitter, "loss": loss, "compliance": compliance,
        "corr": corr,
        "alpha": alpha, "beta": beta, "gamma": gamma, "delta": delta, "A": A, "B": B,
        "w_cvss": w_cvss, "w_expl": w_expl, "w_patch": w_patch, "w_expo": w_expo,
        "w_lat": w_lat, "w_jit": w_jit, "w_loss": w_loss
    }


# ============================================================
# 2) BUILD QUBO
# ============================================================
def build_qubo(problem):
    n = problem["n"]
    K = problem["K"]
    groups = problem["groups"]

    cvss = problem["cvss"]
    exploit_prob = problem["exploit_prob"]
    patch_lag = problem["patch_lag"]
    exposure = problem["exposure"]
    latency = problem["latency"]
    jitter = problem["jitter"]
    loss = problem["loss"]
    compliance = problem["compliance"]
    corr = problem["corr"]

    alpha = problem["alpha"]
    beta = problem["beta"]
    gamma = problem["gamma"]
    delta = problem["delta"]
    A = problem["A"]
    B = problem["B"]

    w_cvss = problem["w_cvss"]
    w_expl = problem["w_expl"]
    w_patch = problem["w_patch"]
    w_expo = problem["w_expo"]
    w_lat = problem["w_lat"]
    w_jit = problem["w_jit"]
    w_loss = problem["w_loss"]

    # score lineari
    risk_intrinsic = w_cvss*cvss + w_expl*exploit_prob + w_patch*patch_lag + w_expo*exposure
    sla_cost = w_lat*latency + w_jit*jitter + w_loss*loss
    linear_base = alpha*risk_intrinsic + gamma*sla_cost + delta*compliance

    # Q simmetrica
    Q = np.zeros((n, n), dtype=float)
    c = 0.0

    # lineari base
    for i in range(n):
        Q[i, i] += linear_base[i]

    # correlazioni
    for i in range(n):
        for j in range(i + 1, n):
            coeff = beta * corr[i, j]
            Q[i, j] += coeff / 2.0
            Q[j, i] += coeff / 2.0

    # cardinalità A*(sum x - K)^2
    for i in range(n):
        Q[i, i] += A * (1 - 2*K)
    c += A * (K**2)
    for i in range(n):
        for j in range(i + 1, n):
            coeff = 2*A
            Q[i, j] += coeff / 2.0
            Q[j, i] += coeff / 2.0

    # anti-SPoF B*(sum_{i in g} x_i - 1)^2
    for g in groups:
        if len(g) <= 1:
            continue
        for i in g:
            Q[i, i] += B * (1 - 2*1)
        c += B * 1
        for a in range(len(g)):
            for b in range(a + 1, len(g)):
                i, j = g[a], g[b]
                coeff = 2*B
                Q[i, j] += coeff / 2.0
                Q[j, i] += coeff / 2.0

    return Q, c


# ============================================================
# 3) QUBO -> SparsePauliOp
# ============================================================
def qubo_to_sparse_pauli(Q: np.ndarray, c: float = 0.0, tol: float = 1e-12) -> SparsePauliOp:
    Q = np.array(Q, dtype=float)
    n = Q.shape[0]
    Qs = 0.5 * (Q + Q.T)

    const = float(c)
    h = np.zeros(n, dtype=float)
    Jzz = np.zeros((n, n), dtype=float)

    for i in range(n):
        qii = Qs[i, i]
        const += 0.5 * qii
        h[i] += -0.5 * qii

    for i in range(n):
        for j in range(i + 1, n):
            bij = 2.0 * Qs[i, j]
            const += 0.25 * bij
            h[i] += -0.25 * bij
            h[j] += -0.25 * bij
            Jzz[i, j] += 0.25 * bij

    paulis, coeffs = [], []

    if abs(const) > tol:
        paulis.append("I" * n)
        coeffs.append(const)

    for i in range(n):
        if abs(h[i]) > tol:
            label = ["I"] * n
            label[n - 1 - i] = "Z"
            paulis.append("".join(label))
            coeffs.append(h[i])

    for i in range(n):
        for j in range(i + 1, n):
            if abs(Jzz[i, j]) > tol:
                label = ["I"] * n
                label[n - 1 - i] = "Z"
                label[n - 1 - j] = "Z"
                paulis.append("".join(label))
                coeffs.append(Jzz[i, j])

    if not paulis:
        paulis, coeffs = ["I"*n], [0.0]

    return SparsePauliOp(paulis, coeffs=np.array(coeffs, dtype=float))


# ============================================================
# 4) Solve (VQE-like + Top3 classico)
# ============================================================
def solve_from_json(path_json: str, random_trials: int = 80):
    p = load_problem(path_json)
    Q, c = build_qubo(p)
    H = qubo_to_sparse_pauli(Q, c)

    n = p["n"]
    ansatz = EfficientSU2(n, entanglement="linear", reps=2)
    backend = AerSimulator()
    estimator = Estimator()

    t_ansatz = transpile(ansatz, backend=backend, optimization_level=1)
    obs = H.apply_layout(t_ansatz.layout)

    def energy(theta):
        job = estimator.run([(t_ansatz, obs, theta)])
        return float(job.result()[0].data.evs)

    best_e = float("inf")
    best_theta = None
    for _ in range(random_trials):
        th = np.random.uniform(0, 2*np.pi, ansatz.num_parameters)
        e = energy(th)
        if e < best_e:
            best_e = e
            best_theta = th

    def qubo_cost(bits):
        x = np.array(bits, dtype=float)
        return float(x @ Q @ x + c)

    all_bits = list(product([0, 1], repeat=n))
    scored = sorted([(qubo_cost(b), b) for b in all_bits], key=lambda t: t[0])

    top3 = scored[:10]
    feasible = [item for item in scored if sum(item[1]) == p["K"]]
    best_feasible = feasible[0] if feasible else None

    print(f"Best estimated quantum energy: {best_e:.6f}")
    print("\nTop-3 configurazioni low-risk:")
    for k, (cost, bits) in enumerate(top3, start=1):
        selected = [i for i, b in enumerate(bits) if b == 1]
        print(f"{k}) bits={bits}, selected={selected}, cost={cost:.6f}")

    if best_feasible:
        cost, bits = best_feasible
        selected = [i for i, b in enumerate(bits) if b == 1]
        print("\nBest feasible (sum x = K):")
        print(f"bits={bits}, selected={selected}, cost={cost:.6f}")


if __name__ == "__main__":
    # Cambia con il tuo file
    solve_from_json("/kaggle/working/cyber_input_large.json", random_trials=80)

/tmp/ipykernel_55/735624949.py:237: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(n, entanglement="linear", reps=2)


CircuitTooWideForTarget: 'Number of qubits (1200) in EfficientSU2 is greater than maximum (30) in the coupling_map'

**correzione codice in numero Qbit virtuali**

In [ ]:
import json
import numpy as np
from itertools import product
from collections import defaultdict
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit import transpile

# ==========================================================
# Load JSON
# ==========================================================
def load_problem(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    channels = sorted(data["channels"], key=lambda x: x["id"])
    n = len(channels)

    ids = [c["id"] for c in channels]
    if ids != list(range(n)):
        raise ValueError("IDs non consecutivi 0..N-1")

    constraints = data["constraints"]
    K = int(constraints["K"])
    w = constraints["weights"]
    iw = constraints.get("internal_weights", {})

    alpha = float(w["alpha"]); beta = float(w["beta"]); gamma = float(w["gamma"]); delta = float(w["delta"])
    A = float(w["A"]); B = float(w["B"])

    w_cvss = float(iw.get("w_cvss", 0.35))
    w_expl = float(iw.get("w_expl", 0.25))
    w_patch = float(iw.get("w_patch", 0.20))
    w_expo = float(iw.get("w_expo", 0.20))
    w_lat = float(iw.get("w_lat", 0.5))
    w_jit = float(iw.get("w_jit", 0.3))
    w_loss = float(iw.get("w_loss", 0.2))

    def arr(key, default=0.0):
        return np.array([float(c.get(key, default)) for c in channels], dtype=float)

    cvss = arr("cvss")
    exploit_prob = arr("exploit_prob")
    patch_lag = arr("patch_lag")
    exposure = arr("exposure")
    latency = arr("latency")
    jitter = arr("jitter")
    loss = arr("loss")
    compliance = arr("compliance")
    groups_labels = [c["group"] for c in channels]

    group_map = defaultdict(list)
    for i, g in enumerate(groups_labels):
        group_map[g].append(i)
    groups = list(group_map.values())

    corr = np.zeros((n, n), dtype=float)
    seen = set()
    for e in data["correlations"]:
        i = int(e["i"]); j = int(e["j"]); s = float(e["score"])
        if not (0 <= i < n and 0 <= j < n and i < j):
            continue
        if (i, j) in seen:
            continue
        seen.add((i, j))
        corr[i, j] = s
        corr[j, i] = s

    return {
        "n": n, "K": K, "groups": groups, "groups_labels": groups_labels,
        "cvss": cvss, "exploit_prob": exploit_prob, "patch_lag": patch_lag, "exposure": exposure,
        "latency": latency, "jitter": jitter, "loss": loss, "compliance": compliance, "corr": corr,
        "alpha": alpha, "beta": beta, "gamma": gamma, "delta": delta, "A": A, "B": B,
        "w_cvss": w_cvss, "w_expl": w_expl, "w_patch": w_patch, "w_expo": w_expo,
        "w_lat": w_lat, "w_jit": w_jit, "w_loss": w_loss
    }

# ==========================================================
# Build QUBO
# ==========================================================
def build_qubo(p):
    n = p["n"]
    K = p["K"]
    groups = p["groups"]

    risk_intrinsic = (
        p["w_cvss"] * p["cvss"] +
        p["w_expl"] * p["exploit_prob"] +
        p["w_patch"] * p["patch_lag"] +
        p["w_expo"] * p["exposure"]
    )
    sla_cost = p["w_lat"] * p["latency"] + p["w_jit"] * p["jitter"] + p["w_loss"] * p["loss"]
    linear_base = p["alpha"] * risk_intrinsic + p["gamma"] * sla_cost + p["delta"] * p["compliance"]

    Q = np.zeros((n, n), dtype=float)
    c = 0.0

    # lineari
    for i in range(n):
        Q[i, i] += linear_base[i]

    # correlazioni
    beta = p["beta"]
    corr = p["corr"]
    for i in range(n):
        for j in range(i + 1, n):
            coeff = beta * corr[i, j]
            if coeff != 0.0:
                Q[i, j] += coeff / 2.0
                Q[j, i] += coeff / 2.0

    # cardinalità
    A = p["A"]; K = p["K"]
    for i in range(n):
        Q[i, i] += A * (1 - 2 * K)
    c += A * (K ** 2)
    for i in range(n):
        for j in range(i + 1, n):
            coeff = 2 * A
            Q[i, j] += coeff / 2.0
            Q[j, i] += coeff / 2.0

    # anti-SPoF group
    B = p["B"]
    for g in groups:
        if len(g) <= 1:
            continue
        for i in g:
            Q[i, i] += B * (1 - 2 * 1)
        c += B
        for a in range(len(g)):
            for b in range(a + 1, len(g)):
                i, j = g[a], g[b]
                coeff = 2 * B
                Q[i, j] += coeff / 2.0
                Q[j, i] += coeff / 2.0

    return Q, c

# ==========================================================
# Helpers
# ==========================================================
def qubo_cost(x, Q, c):
    x = np.asarray(x, dtype=float)
    return float(x @ Q @ x + c)

def qubo_to_sparse_pauli(Qb, cb=0.0, tol=1e-12):
    Qb = np.asarray(Qb, dtype=float)
    n = Qb.shape[0]
    Qs = 0.5 * (Qb + Qb.T)

    const = float(cb)
    h = np.zeros(n, dtype=float)
    Jzz = np.zeros((n, n), dtype=float)

    for i in range(n):
        qii = Qs[i, i]
        const += 0.5 * qii
        h[i] += -0.5 * qii

    for i in range(n):
        for j in range(i + 1, n):
            bij = 2.0 * Qs[i, j]
            const += 0.25 * bij
            h[i] += -0.25 * bij
            h[j] += -0.25 * bij
            Jzz[i, j] += 0.25 * bij

    paulis, coeffs = [], []
    if abs(const) > tol:
        paulis.append("I" * n)
        coeffs.append(const)

    for i in range(n):
        if abs(h[i]) > tol:
            label = ["I"] * n
            label[n - 1 - i] = "Z"
            paulis.append("".join(label))
            coeffs.append(h[i])

    for i in range(n):
        for j in range(i + 1, n):
            if abs(Jzz[i, j]) > tol:
                label = ["I"] * n
                label[n - 1 - i] = "Z"
                label[n - 1 - j] = "Z"
                paulis.append("".join(label))
                coeffs.append(Jzz[i, j])

    if not paulis:
        paulis, coeffs = ["I" * n], [0.0]

    return SparsePauliOp(paulis, coeffs=np.array(coeffs, dtype=float))

def make_blocks(n_vars, block_size=24):
    idx = np.arange(n_vars)
    return [idx[i:i + block_size] for i in range(0, n_vars, block_size)]

def subqubo_with_context(Q, x_fixed, block):
    n = Q.shape[0]
    block = np.asarray(block, dtype=int)
    mask = np.ones(n, dtype=bool)
    mask[block] = False
    rest = np.where(mask)[0]

    QBB = Q[np.ix_(block, block)].copy()
    if len(rest) > 0:
        xr = x_fixed[rest]
        d = 2.0 * (Q[np.ix_(block, rest)] @ xr)
        for i in range(len(block)):
            QBB[i, i] += d[i]
    return QBB, 0.0

def solve_block(QB, brute_force_limit=20, vqe_trials=24):
    m = QB.shape[0]

    if m <= brute_force_limit:
        best_val = float("inf")
        best_bits = None
        for bits in product([0, 1], repeat=m):
            x = np.array(bits, dtype=float)
            v = float(x @ QB @ x)
            if v < best_val:
                best_val = v
                best_bits = x
        return best_bits.astype(int), best_val

    # VQE local
    H = qubo_to_sparse_pauli(QB, 0.0)
    backend = AerSimulator()
    estimator = Estimator()
    ansatz = EfficientSU2(m, entanglement="linear", reps=1)  # reps=1 più leggero
    t_ansatz = transpile(ansatz, backend=backend, optimization_level=1)
    obs = H.apply_layout(t_ansatz.layout)

    def energy(theta):
        job = estimator.run([(t_ansatz, obs, theta)])
        return float(job.result()[0].data.evs)

    best_e = float("inf")
    for _ in range(vqe_trials):
        th = np.random.uniform(0, 2 * np.pi, ansatz.num_parameters)
        e = energy(th)
        if e < best_e:
            best_e = e

    # decode pragmatico: local random rounding
    best_bits = None
    best_val = float("inf")
    for _ in range(200):
        x = np.random.randint(0, 2, size=m).astype(float)
        v = float(x @ QB @ x)
        if v < best_val:
            best_val = v
            best_bits = x

    return best_bits.astype(int), best_val

# ==========================================================
# Feasibility repair: enforce sum(x)=K with greedy flips
# ==========================================================
def enforce_cardinality(x, Q, c, K):
    x = x.copy().astype(int)
    n = len(x)

    def delta_flip(i):
        old = qubo_cost(x, Q, c)
        x[i] = 1 - x[i]
        new = qubo_cost(x, Q, c)
        x[i] = 1 - x[i]
        return new - old

    s = int(np.sum(x))
    while s > K:
        ones = np.where(x == 1)[0]
        best_i = min(ones, key=delta_flip)  # flip 1->0 con minimo danno
        x[best_i] = 0
        s -= 1
    while s < K:
        zeros = np.where(x == 0)[0]
        best_i = min(zeros, key=delta_flip) # flip 0->1 con minimo danno
        x[best_i] = 1
        s += 1
    return x

# ==========================================================
# Main hybrid solve
# ==========================================================
def hybrid_solve(Q, c, K, block_size=24, sweeps=3, seed=42):
    np.random.seed(seed)
    n = Q.shape[0]
    x = np.random.randint(0, 2, size=n).astype(float)
    x = enforce_cardinality(x, Q, c, K).astype(float)

    blocks = make_blocks(n, block_size=block_size)
    prev = qubo_cost(x, Q, c)
    print(f"Initial cost: {prev:.6f} | n={n} | blocks={len(blocks)}")

    for s in range(sweeps):
        for b in blocks:
            QB, _ = subqubo_with_context(Q, x, b)
            xb, _ = solve_block(QB, brute_force_limit=20, vqe_trials=20)
            x[b] = xb

        x = enforce_cardinality(x, Q, c, K).astype(float)
        cur = qubo_cost(x, Q, c)
        print(f"Sweep {s+1}/{sweeps} cost: {cur:.6f}")
        if abs(prev - cur) < 1e-8:
            break
        prev = cur

    return x.astype(int), float(prev)

# ==========================================================
# Run
# ==========================================================
if __name__ == "__main__":
    INPUT_JSON = "/kaggle/working/cyber_input_large.json"
    OUTPUT_JSON = "result_large.json"

    p = load_problem(INPUT_JSON)
    Q, c = build_qubo(p)

    x_best, cost_best = hybrid_solve(
        Q, c, K=p["K"],
        block_size=24,   # <= 30 per compatibilità coupling_map
        sweeps=4,
        seed=42
    )

    selected = np.where(x_best == 1)[0].tolist()
    result = {
        "input_file": INPUT_JSON,
        "n_channels": int(p["n"]),
        "K_target": int(p["K"]),
        "selected_count": int(len(selected)),
        "final_cost": float(round(cost_best, 6)),
        "selected_channels": selected
    }

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print("\nDone.")
    print(f"Selected {len(selected)} channels (target K={p['K']}).")
    print(f"Final cost: {cost_best:.6f}")
    print(f"Saved: {OUTPUT_JSON}")

Initial cost: 1734.825543 | n=1200 | blocks=50


/tmp/ipykernel_55/1178496032.py:234: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(m, entanglement="linear", reps=1)  # reps=1 più leggero
